In [ ]:
!pip install -q pandas numpy tqdm pyarrow sentence-transformers hnswlib

In [ ]:
import os
import json
import numpy as np
import pandas as pd
import hnswlib
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from google.colab import drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================
# CONFIG
# ============================================================

INPUT_DIR = "data/intermediate/topic_top8000_relevance_stance_labeled"
OUTPUT_DIR = "data/intermediate/topic_clean_balanced_diverse_selected"
ARTIFACT_OUTPUT_DIR = "data/intermediate/topic_clean_balanced_artifacts"

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(ARTIFACT_OUTPUT_DIR, exist_ok=True)

CLEAN_FILES = {
    "BLM": "blm_clean_pro_anti.csv",
    "guns": "guns_clean_pro_anti.csv",
    "same_sex_marriage": "samesex_clean_pro_anti.csv",
}

TEXT_COL = "comment_text"

K_TOTAL_PER_TOPIC = 400
RANDOM_SEED = 42

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"

# tuned for your clean datasets
LAMBDA_DIVERSITY = 0.45

In [ ]:
# ============================================================
# LOAD EMBEDDING MODEL
# ============================================================

model = SentenceTransformer(MODEL_NAME)

In [ ]:
# ============================================================
# QUOTA LOGIC: MOST BALANCED POSSIBLE, TOTAL 400 IF POSSIBLE
# ============================================================

def compute_balanced_quotas(df, k_total=400, stance_col="stance_label"):
    counts = df[stance_col].value_counts().to_dict()

    n_pro = int(counts.get("pro", 0))
    n_anti = int(counts.get("anti", 0))

    total_available = n_pro + n_anti

    if total_available < k_total:
        print(f"WARNING: only {total_available} pro/anti rows available, cannot reach {k_total}.")
        return {"pro": n_pro, "anti": n_anti}

    half = k_total // 2

    pro_quota = min(half, n_pro)
    anti_quota = min(half, n_anti)

    remaining = k_total - (pro_quota + anti_quota)

    if remaining > 0:
        pro_left = n_pro - pro_quota
        anti_left = n_anti - anti_quota

        if pro_left >= anti_left:
            add_to_pro = min(remaining, pro_left)
            pro_quota += add_to_pro
            remaining -= add_to_pro

        if remaining > 0:
            add_to_anti = min(remaining, anti_left)
            anti_quota += add_to_anti
            remaining -= add_to_anti

        if remaining > 0:
            pro_left = n_pro - pro_quota
            add_to_pro = min(remaining, pro_left)
            pro_quota += add_to_pro
            remaining -= add_to_pro

    return {"pro": pro_quota, "anti": anti_quota}

In [ ]:
# ============================================================
# DIVERSE + REPRESENTATIVE SELECTION WITHIN ONE STANCE
# ============================================================

def select_diverse_representative_from_embeddings(
    embeddings,
    k,
    lambda_diversity=0.45,
    seed=42,
):
    """
    Select k rows from a clean stance-specific dataset.

    Since your clean stance subsets are only a few thousand rows,
    this version uses all rows as the candidate pool and all rows for coverage.
    No CANDIDATE_POOL_SIZE, COVERAGE_SAMPLE_SIZE, or EVAL_SIZE_PER_STEP is needed.
    """

    rng = np.random.default_rng(seed)

    n = embeddings.shape[0]

    if k >= n:
        return list(range(n))

    emb = np.asarray(embeddings, dtype=np.float32)

    # full similarity matrix: [all rows x all candidates]
    sim_matrix = emb @ emb.T

    selected = []

    current_coverage = np.full(n, -1.0, dtype=np.float32)
    selected_mask = np.zeros(n, dtype=bool)

    # start with the comment closest to the centroid
    center = emb.mean(axis=0)
    center = center / (np.linalg.norm(center) + 1e-12)

    centroid_sims = emb @ center
    first = int(np.argmax(centroid_sims))

    selected.append(first)
    selected_mask[first] = True
    current_coverage = np.maximum(current_coverage, sim_matrix[:, first])

    for _ in tqdm(range(1, k), desc=f"Selecting {k}"):
        available = np.where(~selected_mask)[0]

        candidate_sims_to_dataset = sim_matrix[:, available]
        new_coverage = np.maximum(current_coverage[:, None], candidate_sims_to_dataset)
        coverage_gains = np.sum(new_coverage - current_coverage[:, None], axis=0)

        selected_array = np.array(selected, dtype=int)
        candidate_to_selected_sims = sim_matrix[available][:, selected_array]
        max_sim_to_selected = np.max(candidate_to_selected_sims, axis=1)
        diversity_gains = 1.0 - max_sim_to_selected

        scores = coverage_gains + lambda_diversity * n * diversity_gains

        best_pos = int(np.argmax(scores))
        best_id = int(available[best_pos])

        selected.append(best_id)
        selected_mask[best_id] = True
        current_coverage = np.maximum(current_coverage, sim_matrix[:, best_id])

    return selected

In [ ]:
# ============================================================
# PROCESS ONE TOPIC
# ============================================================

def process_topic(topic_name, input_csv):
    print("\n" + "=" * 80)
    print(f"Processing topic: {topic_name}")
    print("=" * 80)

    df = pd.read_csv(input_csv, encoding="utf-8-sig")

    df["stance_label"] = df["stance_label"].astype(str).str.lower().str.strip()

    df = df[df["stance_label"].isin(["pro", "anti"])].copy()
    df = df.reset_index(drop=True)

    print("Available rows:")
    print(df["stance_label"].value_counts())

    quotas = compute_balanced_quotas(
        df,
        k_total=K_TOTAL_PER_TOPIC,
        stance_col="stance_label"
    )

    print("Selection quotas:", quotas)

    selected_parts = []

    for stance, quota in quotas.items():
        stance_df = df[df["stance_label"] == stance].copy().reset_index(drop=True)

        print(f"\nEmbedding {stance} rows: {len(stance_df)}")
        texts = stance_df[TEXT_COL].fillna("").astype(str).tolist()

        stance_emb = model.encode(
            texts,
            batch_size=256,
            show_progress_bar=True,
            normalize_embeddings=True,
            convert_to_numpy=True,
        ).astype(np.float32)

        print(f"Selecting {quota} diverse {stance} rows")
        local_selected_ids = select_diverse_representative_from_embeddings(
            embeddings=stance_emb,
            k=quota,
            lambda_diversity=LAMBDA_DIVERSITY,
            seed=RANDOM_SEED,
        )

        selected_df = stance_df.iloc[local_selected_ids].copy()
        selected_df["selection_stance_quota"] = quota
        selected_df["selection_stage"] = "balanced_diverse_selection"

        selected_parts.append(selected_df)

    final_df = pd.concat(selected_parts, ignore_index=True)

    final_df = final_df.sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)
    final_df["topic_selection"] = topic_name
    final_df["selection_rank"] = range(1, len(final_df) + 1)

    print("\nFinal selected counts:")
    print(final_df["stance_label"].value_counts())
    print("Total:", len(final_df))

    output_path = os.path.join(
        OUTPUT_DIR,
        f"{topic_name}_selected_{len(final_df)}_balanced_diverse.csv"
    )

    final_df.to_csv(output_path, index=False, encoding="utf-8-sig")
    print("Saved selected CSV:", output_path)

    return final_df

In [ ]:
# ============================================================
# RUN ALL TOPICS
# ============================================================

all_selected = []

for topic_name, filename in CLEAN_FILES.items():
    input_csv = os.path.join(INPUT_DIR, filename)

    selected_df = process_topic(
        topic_name=topic_name,
        input_csv=input_csv
    )

    all_selected.append(selected_df)

combined = pd.concat(all_selected, ignore_index=True)

combined_path = os.path.join(
    OUTPUT_DIR,
    f"all_topics_selected_{len(combined)}_balanced_diverse.csv"
)

combined.to_csv(combined_path, index=False, encoding="utf-8-sig")

print("Saved combined file:", combined_path)
print(combined["topic_selection"].value_counts())
print(combined.groupby(["topic_selection", "stance_label"]).size())